# Annotate merged single cells with metadata from platemap file

## Import libraries

In [1]:
import argparse
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tqdm
from pycytominer import annotate
from pycytominer.cyto_utils import output

try:
    cfg = get_ipython().config
    in_notebook = True
except NameError:
    in_notebook = False

## Set paths and variables

In [2]:
# load in platemap file as a pandas dataframe
platemap_path = pathlib.Path("../../data/").resolve()

# directory where parquet files are located
data_dir = pathlib.Path("../data/1.annotated_data").resolve()

# directory where the annotated parquet files are saved to
profiles_output_dir = pathlib.Path(
    "../data/2.sc_tracks_annotated_data/profiles/endpoint/"
).resolve()
stats_output_dir = pathlib.Path(
    "../data/2.sc_tracks_annotated_data/stats/endpoint/"
).resolve()

profiles_output_dir.mkdir(exist_ok=True, parents=True)
stats_output_dir.mkdir(exist_ok=True, parents=True)

if not in_notebook:
    print("Running as script")
    # set up arg parser
    parser = argparse.ArgumentParser(description="Single cell extraction")

    parser.add_argument(
        "--well_fov",
        type=str,
        help="Path to the input directory containing the tiff images",
    )

    args = parser.parse_args()
    well_fov = args.well_fov
else:
    print("Running in a notebook")
    well_fov = "C-02_F0003"

Running in a notebook


In [3]:
tracks = pathlib.Path(
    f"../../5.cell_tracking/results/{well_fov}_tracks.parquet"
).resolve(strict=True)
profiles = pathlib.Path(
    f"../data/1.annotated_data/endpoint/{well_fov}_sc.parquet"
).resolve(strict=True)

tracks = pd.read_parquet(tracks)
profiles = pd.read_parquet(profiles)
# prepend Metadata_ to the tracks columns
tracks.columns = ["Metadata_" + str(col) for col in tracks.columns]
tracks["Metadata_coordinates"] = list(zip(tracks["Metadata_x"], tracks["Metadata_y"]))
profiles["Metadata_coordinates"] = list(
    zip(profiles["Nuclei_AreaShape_Center_X"], profiles["Nuclei_AreaShape_Center_Y"])
)
# get only the last timepoint for each track
tracks = tracks.loc[tracks["Metadata_t"] == tracks["Metadata_t"].max()]

profiles["Metadata_Time"] = profiles["Metadata_Time"].astype(float)
profiles["Metadata_Time"] = profiles["Metadata_Time"] - 1

In [4]:
print(f"Number of tracks: {len(tracks)}")
print(f"Number of profiles: {len(profiles)}")

Number of tracks: 149
Number of profiles: 158


In [5]:
coordinate_column_left = "Metadata_coordinates"
coordinate_column_right = "Metadata_coordinates"
pixel_cutt_off = 5
left_on = ["Metadata_Time"]
right_on = ["Metadata_t"]
merged_df_list = []  # list to store the merged dataframes
total_CP_cells = 0  # total number of cells in the left dataframe
total_annotated_cells = 0  # total number of cells that were annotated
distances = []  # list to store the distances between the coordinates

In [ ]:
tracked_cells_stats = {
    "Metadata_time": [],  # timepoint of the cell
    "total_CP_cells": [],  # total number of cells segmented
    "total_annotated_cells": [],  # total number of cells tracked
}
time = profiles["Metadata_Time"].unique()[0]
df_left = profiles.copy()
df_right = tracks.copy()

total_CP_cells += df_left.shape[0]

# Keep matching one-to-one: each track row can be used at most once.
used_track_indices = set()
euclidean_cut_off = np.linalg.norm(
    np.array([0, 0]) - np.array([pixel_cutt_off, pixel_cutt_off])
)

# loop through the rows in subset_annotated_df and find the closest unmatched coordinate
for index1, row1 in df_left.iterrows():
    tracked_cells_stats["total_CP_cells"].append(1)

    best_dist = np.inf
    best_index = None
    coord1 = row1[coordinate_column_left]

    for index2, row2 in df_right.iterrows():
        if index2 in used_track_indices:
            continue

        coord2 = row2[coordinate_column_right]
        try:
            temp_dist = np.linalg.norm(np.array(coord1) - np.array(coord2))
        except Exception:
            temp_dist = np.inf

        if temp_dist < best_dist:
            best_dist = temp_dist
            best_index = index2

    if best_index is not None and best_dist < euclidean_cut_off:
        temp_merged_df = pd.merge(
            df_left.loc[[index1]],
            df_right.loc[[best_index]],
            how="inner",
            left_on=left_on,
            right_on=right_on,
        )

        if temp_merged_df.shape[0] > 0:
            used_track_indices.add(best_index)
            distances.append(best_dist)
            total_annotated_cells += temp_merged_df.shape[0]
            tracked_cells_stats["Metadata_time"].append(time)
            # if the cell is tracked and annotated, append 1 to total annotated cells
            tracked_cells_stats["total_annotated_cells"].append(1)
            merged_df_list.append(temp_merged_df)
        else:
            tracked_cells_stats["Metadata_time"].append(time)
            tracked_cells_stats["total_annotated_cells"].append(0)
    else:
        tracked_cells_stats["Metadata_time"].append(time)
        # if the cell is not tracked and annotated, append 0 to total annotated cells
        tracked_cells_stats["total_annotated_cells"].append(0)

if len(merged_df_list) == 0:
    merged_df_list.append(pd.DataFrame())
merged_df = pd.concat(merged_df_list)
merged_df["Metadata_distance"] = distances
tracked_cells_stats = {
    "Metadata_time": [],  # timepoint of the cell
    "total_CP_cells": [],  # total number of cells segmented
    "total_annotated_cells": [],  # total number of cells tracked
}
time = profiles["Metadata_Time"].unique()[0]
df_left = profiles.copy()
df_right = tracks.copy()

total_CP_cells += df_left.shape[0]

# Keep matching one-to-one: each track row can be used at most once.
used_track_indices = set()
euclidean_cut_off = np.linalg.norm(
    np.array([0, 0]) - np.array([pixel_cutt_off, pixel_cutt_off])
)

# loop through the rows in subset_annotated_df and find the closest unmatched coordinate
for index1, row1 in df_left.iterrows():
    tracked_cells_stats["total_CP_cells"].append(1)

    best_dist = np.inf
    best_index = None
    coord1 = row1[coordinate_column_left]

    for index2, row2 in df_right.iterrows():
        if index2 in used_track_indices:
            continue

        coord2 = row2[coordinate_column_right]
        try:
            temp_dist = np.linalg.norm(np.array(coord1) - np.array(coord2))
        except Exception:
            temp_dist = np.inf

        if temp_dist < best_dist:
            best_dist = temp_dist
            best_index = index2

    if best_index is not None and best_dist < euclidean_cut_off:
        temp_merged_df = pd.merge(
            df_left.loc[[index1]],
            df_right.loc[[best_index]],
            how="inner",
            left_on=left_on,
            right_on=right_on,
        )

        if temp_merged_df.shape[0] > 0:
            used_track_indices.add(best_index)
            distances.append(best_dist)
            total_annotated_cells += temp_merged_df.shape[0]
            tracked_cells_stats["Metadata_time"].append(time)
            # if the cell is tracked and annotated, append 1 to total annotated cells
            tracked_cells_stats["total_annotated_cells"].append(1)
            merged_df_list.append(temp_merged_df)
        else:
            tracked_cells_stats["Metadata_time"].append(time)
            tracked_cells_stats["total_annotated_cells"].append(0)
    else:
        tracked_cells_stats["Metadata_time"].append(time)
        # if the cell is not tracked and annotated, append 0 to total annotated cells
        tracked_cells_stats["total_annotated_cells"].append(0)

if len(merged_df_list) == 0:
    merged_df_list.append(pd.DataFrame())
merged_df = pd.concat(merged_df_list)
merged_df["Metadata_distance"] = distances

# replace Metadata string in column names with Metadata (Non Morphology Features)
merged_df.columns = [
    x.replace("Metadata_", "Metadata_") if "Metadata_" in x else x
    for x in merged_df.columns
]

print(f"Annotated cells: {total_annotated_cells} out of {total_CP_cells}")
print(f"Percentage of annotated cells: {total_annotated_cells/total_CP_cells*100}%")
print(merged_df.shape)
merged_df.to_parquet(profiles_output_dir / f"{well_fov}_annotated_tracks.parquet")
merged_df.head()
# replace Metadata string in column names with Metadata (Non Morphology Features)
merged_df.columns = [
    x.replace("Metadata_", "Metadata_") if "Metadata_" in x else x
    for x in merged_df.columns
]

print(f"Annotated cells: {total_annotated_cells} out of {total_CP_cells}")
print(f"Percentage of annotated cells: {total_annotated_cells/total_CP_cells*100}%")
print(merged_df.shape)
merged_df.to_parquet(profiles_output_dir / f"{well_fov}_annotated_tracks.parquet")
merged_df.head()

Annotated cells: 94 out of 158
Percentage of annotated cells: 59.49367088607595%
(94, 1216)


,Metadata_plate,Metadata_Well,Metadata_number_of_singlecells,Metadata_compound,Metadata_dose,Metadata_control,Metadata_ImageNumber,Metadata_FOV,Metadata_Time,Metadata_Cells_Number_Object_Number,...,Metadata_coordinates_x,Metadata_track_id,Metadata_t,Metadata_y,Metadata_x,Metadata_id,Metadata_parent_track_id,Metadata_parent_id,Metadata_coordinates_y,Metadata_distance
0,1,C-02,158,Staurosporine,0.0,negative,1,0003,13.0,2,...,"(50.412921348314605, 83.76638576779027)",6,13.0,84.0,50.0,14000011.0,-1,13000012.0,"(50.0, 84.0)",0.474426
0,1,C-02,158,Staurosporine,0.0,negative,1,0003,13.0,6,...,"(5.425465838509317, 1567.2531055900622)",167,13.0,1567.0,5.0,14000135.0,-1,13000144.0,"(5.0, 1567.0)",0.495059
0,1,C-02,158,Staurosporine,0.0,negative,1,0003,13.0,7,...,"(26.94582005393479, 1755.2522677126747)",136,13.0,1755.0,27.0,14000157.0,-1,13000165.0,"(27.0, 1755.0)",0.258020
0,1,C-02,158,Staurosporine,0.0,negative,1,0003,13.0,11,...,"(32.625068418171864, 1136.8159095055646)",20,13.0,1137.0,33.0,14000090.0,-1,13000100.0,"(33.0, 1137.0)",0.417688
0,1,C-02,158,Staurosporine,0.0,negative,1,0003,13.0,12,...,"(116.81004250797024, 994.1702975557918)",95,13.0,994.0,117.0,14000080.0,-1,13000089.0,"(117.0, 994.0)",0.255118


In [8]:
# get the number of tracks for each track length
list_of_track_lengths = []
for track in merged_df["Metadata_track_id"].unique():
    track_length = merged_df.loc[merged_df["Metadata_track_id"] == track].shape[0]
    list_of_track_lengths.append(track_length)
list_of_track_lengths_df = pd.DataFrame(list_of_track_lengths, columns=["track_length"])
list_of_track_lengths_df = (
    list_of_track_lengths_df.value_counts().to_frame().reset_index()
)
list_of_track_lengths_df["well_fov"] = well_fov
# save the list of track lengths to a parquet file
list_of_track_lengths_df.to_parquet(
    stats_output_dir / f"{well_fov}_track_lengths.parquet"
)

In [9]:
# save the tracked cells stats to a parquet file
tracked_cells_stats_df = pd.DataFrame(tracked_cells_stats)
tracked_cells_stats_df["well_fov"] = well_fov

In [10]:
# get the number of cells for each time point
tracked_cells_stats_df = (
    tracked_cells_stats_df.groupby(["Metadata_time", "well_fov"]).sum().reset_index()
)

In [11]:
# save the stats to a parquet file
tracked_cells_stats_df.to_parquet(stats_output_dir / f"{well_fov}_stats.parquet")